VJ decode diagnostic (CPU, no GPU quota): why does nurikw3's infer.py read only ~1.8 of ~4.7 views per gold study?

    python build.py <oai_gold58_notebook.ipynb>

Calls the author's _load_source_slices directly (load_study_tensor swallows its exceptions) on every VIEWS5 candidate
series of the first 12 gold studies and prints the exception, the transfer syntax and the file count. No labels are read.


In [ ]:
# Stand-in competition root: "test" = the 58 studies of train.csv with expert labels.
import os, glob
import pandas as pd
COMP = next(c for c in ('/kaggle/input/competitions/rsna-knee-abnormality-detection', '/kaggle/input/rsna-knee-abnormality-detection')
            if os.path.exists(c + '/train.csv'))
G = '/kaggle/working/gold_root'
os.makedirs(G + '/test_series', exist_ok=True)
tr = pd.read_csv(COMP + '/train.csv', dtype={'StudyInstanceUID': str})
ss = pd.read_csv(COMP + '/sample_submission.csv', nrows=1)
labels = [c for c in ss.columns if c != 'StudyInstanceUID']
gold = tr[tr[labels].notna().all(axis=1)].reset_index(drop=True)
print('gold studies', len(gold))
gold[['StudyInstanceUID']].to_csv(G + '/test.csv', index=False)
gold[['StudyInstanceUID'] + labels].to_csv(G + '/gold_labels.csv', index=False)
sub = gold[['StudyInstanceUID']].copy()
for c in labels: sub[c] = 0.5
sub.to_csv(G + '/sample_submission.csv', index=False)
ser = pd.read_csv(COMP + '/train_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
ser[ser.StudyInstanceUID.isin(set(gold.StudyInstanceUID))].to_csv(G + '/test_series.csv', index=False)
for u in gold.StudyInstanceUID:
    dst = G + '/test_series/' + u
    if not os.path.exists(dst):
        os.symlink(COMP + '/train_series/' + u, dst)
shutil_ok = True
print('root ready', G, len(os.listdir(G + '/test_series')))


In [ ]:

import glob, sys, traceback, collections
from pathlib import Path
import pandas as pd, pydicom
c = [p for p in glob.glob('/kaggle/input/**/f0_best.pt', recursive=True) if 'vjepa21' in p]
DS = str(Path(c[0]).parent)
sys.path.append(DS)
import infer
print('pydicom', pydicom.__version__)
for m in ('pylibjpeg', 'gdcm', 'libjpeg', 'openjpeg', 'jpeg_ls', 'dicomsdl'):
    try:
        __import__(m); print('has', m)
    except Exception as e:
        print('no', m, type(e).__name__)
comp = Path(G)
ser = pd.read_csv(comp / 'test_series.csv', dtype={'StudyInstanceUID': str})
uids = pd.read_csv(comp / 'test.csv', dtype={'StudyInstanceUID': str}).StudyInstanceUID.tolist()[:12]
errs, ts, ok = collections.Counter(), collections.Counter(), 0
for u in uids:
    g = ser[ser.StudyInstanceUID == u]
    for plane, fluid in infer.VIEWS5:
        for row in g[(g.Anatomical_Plane == plane) & (g.Fluid_Sensitive == fluid)].itertuples(index=False):
            d = comp / 'test_series' / u / str(row.SeriesInstanceUID)
            files = sorted(glob.glob(str(d / '*.dcm')))
            any_files = sorted(glob.glob(str(d / '*')))
            t = None
            if any_files:
                try:
                    t = str(pydicom.dcmread(any_files[0], stop_before_pixels=True, force=True).file_meta.TransferSyntaxUID)
                except Exception as e:
                    t = 'hdr-err ' + type(e).__name__
            ts[t] += 1
            try:
                s = infer._load_source_slices(d)
                r = 'None' if s is None else ('ok %s nonzero %s' % (s.shape, bool(s.any())))
                ok += s is not None and bool(s.any())
            except Exception as e:
                r = 'EXC ' + repr(e)[:160]
                errs[type(e).__name__ + ': ' + str(e)[:80]] += 1
            print(u[-8:], plane, fluid, 'dcm %d all %d' % (len(files), len(any_files)), t, '->', r, flush=True)
            break
print('series ok', ok, 'transfer syntaxes', dict(ts))
print('errors', dict(errs))
print('VJ_DIAG_DONE')
